In [1]:
import pandas as pd
import sqlite3

In [2]:
# Create database
conn = sqlite3.connect("output/scraped_data.db")
cursor = conn.cursor()

# Create table
create_tab_statement = """
CREATE TABLE IF NOT EXISTS nharieng(
    "Tỉnh/Thành phố" TEXT,
    "Thành phố/Quận/Huyện/Thị xã" TEXT,
    "Xã/Phường/Thị trấn" TEXT,
    "Đường phố" TEXT,
    "Chi tiết" TEXT,
    "Nguồn thông tin" TEXT,
    "Tình trạng giao dịch" TEXT,
    "Thời điểm giao dịch/rao bán" DATE,
    "Thông tin liên hệ" TEXT,
    "Giá rao bán/giao dịch" INTEGER,
    "Giá ước tính" INTEGER,
    "Loại đơn giá (đ/m2 hoặc đ/m ngang)" TEXT,
    "Đơn giá đất" REAL,
    "Lợi thế kinh doanh" TEXT,
    "Số tầng công trình" REAL,
    "Tổng diện tích sàn" REAL,
    "Đơn giá xây dựng" REAL,
    "Năm xây dựng" INTEGER,
    "Chất lượng còn lại" REAL,
    "Diện tích đất (m2)" REAL,
    "Kích thước mặt tiền (m)" REAL,
    "Kích thước chiều dài (m)" REAL,
    "Số mặt tiền tiếp giáp" INTEGER,
    "Hình dạng" TEXT,
    "Độ rộng ngõ/ngách nhỏ nhất (m)" REAL,
    "Khoảng cách tới trục đường chính (m)" REAL,
    "Mục đích sử dụng đất" TEXT,
    "Yếu tố khác" TEXT,
    "Tọa độ (vĩ độ)" REAL,
    "Tọa độ (kinh độ)" REAL,
    "Hình ảnh của bài đăng" TEXT,
    Web TEXT
);"""

create_unique_index = """
CREATE UNIQUE INDEX IF NOT EXISTS unique_index
ON nharieng(
            "Tỉnh/Thành phố",  
            "Thành phố/Quận/Huyện/Thị xã",  
            "Xã/Phường/Thị trấn",  
            "Đường phố",  
            "Giá rao bán/giao dịch",  
            "Số tầng công trình",  
            "Tổng diện tích sàn",  
            "Đơn giá xây dựng",  
            "Chất lượng còn lại",  
            "Diện tích đất (m2)",  
            "Kích thước mặt tiền (m)",  
            "Kích thước chiều dài (m)",  
            "Số mặt tiền tiếp giáp",  
            "Hình dạng",  
            "Độ rộng ngõ/ngách nhỏ nhất (m)",  
            "Khoảng cách tới trục đường chính (m)",  
            "Mục đích sử dụng đất");
"""

cursor.execute(create_tab_statement)
cursor.execute(create_unique_index)
conn.commit()
conn.close()

In [3]:
import pandas as pd

onehousing = pd.read_excel('output/Onehousing/30.12.2025-06.01.2026.xlsx')
onehousing.columns

Index(['Tỉnh/Thành phố', 'Thành phố/Quận/Huyện/Thị xã', 'Xã/Phường/Thị trấn',
       'Đường phố', 'Nguồn thông tin', 'Tình trạng giao dịch',
       'Thời điểm giao dịch/rao bán', 'Thông tin liên hệ',
       'Giá rao bán/giao dịch', 'Giá ước tính',
       'Loại đơn giá (đ/m2 hoặc đ/m ngang)', 'Đơn giá đất',
       'Số tầng công trình', 'Chất lượng còn lại', 'Đơn giá xây dựng',
       'Diện tích đất (m2)', 'Tổng diện tích sàn', 'Kích thước mặt tiền (m)',
       'Kích thước chiều dài (m)', 'Số mặt tiền tiếp giáp', 'Hình dạng',
       'Độ rộng ngõ/ngách nhỏ nhất (m)',
       'Khoảng cách tới trục đường chính (m)', 'Mục đích sử dụng đất',
       'Hình ảnh của bài đăng', 'Yếu tố khác'],
      dtype='object')

In [5]:
bds_path = 'output/07.11.2025-08.01.2026.xlsx'
onehousing_path =  'output/Onehousing/30.12.2025-06.01.2026.xlsx'

def insert_or_ignore(table, conn, keys, data_iter):
    quoted_keys = [f'"{k}"' for k in keys]
    sql = (
        f'INSERT OR IGNORE INTO "{table.name}" '
        f'({",".join(quoted_keys)}) '
        f'VALUES ({",".join(["?"] * len(keys))})'
    )
    return conn.executemany(sql, data_iter)

with sqlite3.connect("output/scraped_data.db") as conn:
    bds_df = pd.read_excel(bds_path)
    bds_df['Web'] = 'Batdongsan'
    onehousing_df = pd.read_excel(onehousing_path)
    onehousing_df['Web'] = 'Onehousing'
    print(bds_df.columns)

    bds_df.to_sql('nharieng', conn, if_exists='append', index=False, method=insert_or_ignore)
    onehousing_df.to_sql('nharieng', conn, if_exists='append', index=False, method=insert_or_ignore)
    
    cursor = conn.cursor()
    cursor.execute("SELECT COUNT(*) FROM nharieng;")
    n_rows = cursor.fetchone()[0]
    print(n_rows)
    # Drop duplicated values
    # inspect_dup = """SELECT *, COUNT(*) FROM nharieng
    #               GROUP BY  Tỉnh/Thành phố ,  Thành phố/Quận/Huyện/Thị xã ,  Xã/Phường/Thị trấn ,  Đường phố ,  Giá rao bán/giao dịch ,  Giá ước tính ,  Đơn giá đất ,  Lợi thế kinh doanh ,  Số tầng công trình ,  Tổng diện tích sàn ,  Đơn giá xây dựng ,  Chất lượng còn lại ,  Diện tích đất (m2) ,  Kích thước mặt tiền (m) ,  Kích thước chiều dài (m) ,  Số mặt tiền tiếp giáp ,  Hình dạng ,  Độ rộng ngõ/ngách nhỏ nhất (m) ,  Khoảng cách tới trục đường chính (m) ,  Mục đích sử dụng đất
    #               HAVING COUNT(*) > 1"""
    # delete_dup = """DELETE FROM nharieng
    #                 WHERE rowid NOT IN (
    #                 SELECT MIN(rowid)
    #                 FROM nharieng
    #                 GROUP BY "Tỉnh/Thành phố",  
    #                         "Thành phố/Quận/Huyện/Thị xã",  
    #                         "Xã/Phường/Thị trấn",  
    #                         "Đường phố",  
    #                         "Giá rao bán/giao dịch",  
    #                         "Giá ước tính",  
    #                         "Đơn giá đất",  
    #                         "Lợi thế kinh doanh",  
    #                         "Số tầng công trình",  
    #                         "Tổng diện tích sàn",  
    #                         "Đơn giá xây dựng",  
    #                         "Chất lượng còn lại",  
    #                         "Diện tích đất (m2)",  
    #                         "Kích thước mặt tiền (m)",  
    #                         "Kích thước chiều dài (m)",  
    #                         "Số mặt tiền tiếp giáp",  
    #                         "Hình dạng",  
    #                         "Độ rộng ngõ/ngách nhỏ nhất (m)",  
    #                         "Khoảng cách tới trục đường chính (m)",  
    #                         "Mục đích sử dụng đất")"""

Index(['Tỉnh/Thành phố', 'Thành phố/Quận/Huyện/Thị xã', 'Xã/Phường/Thị trấn',
       'Đường phố', 'Chi tiết', 'Nguồn thông tin', 'Tình trạng giao dịch',
       'Thời điểm giao dịch/rao bán', 'Thông tin liên hệ',
       'Giá rao bán/giao dịch', 'Giá ước tính',
       'Loại đơn giá (đ/m2 hoặc đ/m ngang)', 'Đơn giá đất',
       'Lợi thế kinh doanh', 'Số tầng công trình', 'Tổng diện tích sàn',
       'Đơn giá xây dựng', 'Năm xây dựng', 'Chất lượng còn lại',
       'Diện tích đất (m2)', 'Kích thước mặt tiền (m)',
       'Kích thước chiều dài (m)', 'Số mặt tiền tiếp giáp', 'Hình dạng',
       'Độ rộng ngõ/ngách nhỏ nhất (m)',
       'Khoảng cách tới trục đường chính (m)', 'Mục đích sử dụng đất',
       'Yếu tố khác', 'Tọa độ (vĩ độ)', 'Tọa độ (kinh độ)',
       'Hình ảnh của bài đăng', 'Web'],
      dtype='object')
8433


In [11]:
bds_df.shape[0] + onehousing_df.shape[0]

8435

In [7]:
with sqlite3.connect('output/scraped_data.db') as conn:
    df = pd.read_sql('SELECT * FROM nharieng', conn)
    # cursor = conn.cursor()
    # sql = 'SELECT * FROM nharieng'
    # cursor.execute(sql)
    # result = cursor.fetchmany(n_rows)
    # print(len(result))

In [8]:
for col in df.columns:
    print(f'NaN values for column {col}: {df[col].isna().sum()} (/{df.shape[0]})')

NaN values for column Tỉnh/Thành phố: 0 (/8433)
NaN values for column Thành phố/Quận/Huyện/Thị xã: 0 (/8433)
NaN values for column Xã/Phường/Thị trấn: 0 (/8433)
NaN values for column Đường phố: 0 (/8433)
NaN values for column Chi tiết: 1482 (/8433)
NaN values for column Nguồn thông tin: 0 (/8433)
NaN values for column Tình trạng giao dịch: 0 (/8433)
NaN values for column Thời điểm giao dịch/rao bán: 1482 (/8433)
NaN values for column Thông tin liên hệ: 8433 (/8433)
NaN values for column Giá rao bán/giao dịch: 0 (/8433)
NaN values for column Giá ước tính: 0 (/8433)
NaN values for column Loại đơn giá (đ/m2 hoặc đ/m ngang): 0 (/8433)
NaN values for column Đơn giá đất: 1482 (/8433)
NaN values for column Lợi thế kinh doanh: 1482 (/8433)
NaN values for column Số tầng công trình: 0 (/8433)
NaN values for column Tổng diện tích sàn: 0 (/8433)
NaN values for column Đơn giá xây dựng: 0 (/8433)
NaN values for column Năm xây dựng: 8433 (/8433)
NaN values for column Chất lượng còn lại: 0 (/8433)
NaN

In [ ]:
for col in bds_df.columns:
    print(f'NaN values for column {col}: {bds_df[col].isna().sum()} (/{bds_df.shape[0]})')

NaN values for column Tỉnh/Thành phố: 0 (/6951)
NaN values for column Thành phố/Quận/Huyện/Thị xã: 0 (/6951)
NaN values for column Xã/Phường/Thị trấn: 0 (/6951)
NaN values for column Đường phố: 0 (/6951)
NaN values for column Chi tiết: 0 (/6951)
NaN values for column Nguồn thông tin: 0 (/6951)
NaN values for column Tình trạng giao dịch: 0 (/6951)
NaN values for column Thời điểm giao dịch/rao bán: 0 (/6951)
NaN values for column Thông tin liên hệ: 6951 (/6951)
NaN values for column Giá rao bán/giao dịch: 0 (/6951)
NaN values for column Giá ước tính: 0 (/6951)
NaN values for column Loại đơn giá (đ/m2 hoặc đ/m ngang): 0 (/6951)
NaN values for column Đơn giá đất: 0 (/6951)
NaN values for column Lợi thế kinh doanh: 0 (/6951)
NaN values for column Số tầng công trình: 0 (/6951)
NaN values for column Tổng diện tích sàn: 0 (/6951)
NaN values for column Đơn giá xây dựng: 0 (/6951)
NaN values for column Năm xây dựng: 6951 (/6951)
NaN values for column Chất lượng còn lại: 0 (/6951)
NaN values for 

In [12]:
import sqlite3

new_bds=pd.read_excel(bds_path)

cols = list(new_bds.columns)
placeholders = ",".join(["?"] * len(cols))
quoted_cols = ",".join(f'"{c}"' for c in cols)

sql = f"""
INSERT OR IGNORE INTO nharieng ({quoted_cols})
VALUES ({placeholders})
"""

with sqlite3.connect("output/scraped_data.db") as conn:
    conn.executemany(sql, new_bds.itertuples(index=False, name=None))
    conn.commit()

    cursor = conn.cursor()
    cursor.execute("SELECT COUNT(*) FROM nharieng;")
    n_rows = cursor.fetchone()[0]
    print(n_rows)

8433


In [3]:
import sqlite3
from datetime import datetime
import time

start_date = datetime("2025-11-10 00:00:00").date()
end_date = datetime('2025-12-25 23:59:59').date()

start_time = time.time()
with sqlite3.connect('output/scrape_data.db') as conn:
    cursor = conn.cursor()
    sql = f"""
SELECT * FROM nharieng 
HAVING (
"Thời điểm giao dịch/rao bán" >= {start_date}
AND "Thời điểm giao dịch/rao bán" <= {end_date})
"""
    cursor.execute(sql)
    df = cursor.fetchall()
end_time = time.time()
print(end_time-start_time)

TypeError: 'str' object cannot be interpreted as an integer

In [22]:
import sqlite3
from datetime import datetime
import time
import pandas as pd

start_date = datetime.strptime("10-11-2025 00:00:00", "%d-%m-%Y %H:%M:%S").date()
end_date   = datetime.strptime("25-12-2025 23:59:59", "%d-%m-%Y %H:%M:%S").date()

start_time = time.time()

with sqlite3.connect("output/scraped_data.db") as conn:
    cursor = conn.cursor()

    sql = """
    SELECT *
    FROM nharieng
    WHERE
    date(
        substr("Thời điểm giao dịch/rao bán", 7, 4) || '-' ||
        substr("Thời điểm giao dịch/rao bán", 4, 2) || '-' ||
        substr("Thời điểm giao dịch/rao bán", 1, 2)
    )
    BETWEEN date(?) AND date(?)
    """

    cursor.execute(
        sql,
        (
            '2025-11-10',
            '2025-12-25',
        )
    )

    rows = cursor.fetchall()

end_time = time.time()
print("Elapsed SQL:", end_time - start_time)
print("Rows SQL:", len(rows))

start_time = time.time()

start_date = pd.Timestamp("2025-11-10")
end_date   = pd.Timestamp("2025-12-25")

with sqlite3.connect('output/scraped_data.db') as conn:
    df = pd.read_sql(f'SELECT * FROM nharieng', conn)
df['Time'] = pd.to_datetime(df['Thời điểm giao dịch/rao bán'], dayfirst=True)
output_df = df[(df['Time'] <= end_date) & (df['Time'] >= start_date)]
output_df.drop(columns='Time', inplace=True)
end_time = time.time()
print("Elapsed Pandas:", end_time - start_time)
print("Rows Pandas:", len(rows))

Elapsed SQL: 0.02676248550415039
Rows SQL: 1896
Elapsed Pandas: 0.08634424209594727
Rows Pandas: 1896


C:\Users\OS TDTS\AppData\Local\Temp\ipykernel_8304\1069340603.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  output_df.drop(columns='Time', inplace=True)


In [16]:
sql_df = pd.DataFrame(rows, columns = output_df.columns)

In [24]:
sql_df.drop(columns=['Thông tin liên hệ', 'Năm xây dựng'], inplace=True)
output_df.drop(columns=['Thông tin liên hệ', 'Năm xây dựng'], inplace=True)

C:\Users\OS TDTS\AppData\Local\Temp\ipykernel_8304\953401161.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  output_df.drop(columns=['Thông tin liên hệ', 'Năm xây dựng'], inplace=True)


In [30]:
different = []

for row in range(output_df.shape[0]):
    if output_df.iloc[row]['Thời điểm giao dịch/rao bán'] != sql_df.iloc[row]['Thời điểm giao dịch/rao bán']:
        different.append(row)

In [31]:
different

[]